# Echo State Network as a tunable frequency generator

This is a simplified implementation of Herbert Jaeger's task of learning a simple non-autonomous system, [a frequency generator controlled by an external signal](http://www.scholarpedia.org/article/Echo_state_network). Plots at the end.

[See the ESN implementation](https://github.com/cknd/pyESN).


In [1]:
import numpy as np
%matplotlib notebook
from matplotlib import pyplot as plt
%matplotlib inline
from pyESN import ESN
from sklearn.model_selection import train_test_split
from Loader import Loader
import torch
import os
import pickle
from tqdm import tqdm

In [2]:
random_state = 42

## Task

The network will learn to generate a wave signal whose frequency is determined by some slowly changing control input.

#### 1) Generate some sample data:


In [3]:
threshold = 1000

def get_mask_numpy(u, y):
    nan_mask = np.isnan(u).any(axis=(1, 2)) | np.isnan(y).any(axis=(1, 2))
    nan_samples_count = np.sum(nan_mask)

    max_y = y.max(axis=2).max(axis=1)
    threshold_mask = max_y <= threshold
    large_samples_count = np.sum(~threshold_mask)

    valid_mask = ~nan_mask & threshold_mask
    return valid_mask, nan_samples_count, large_samples_count

In [4]:
def load_file(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return None

In [5]:
!ls '../NIEMRE_Feb20_2025/Hodgkin_Huxley'

config.sh  hh_model_long_local.sh     HHSimulator.py	  run_HH.sh
data	   hh_model_long.sh	      logfiles		  run_local.sh
debug.py   hh_model.sh		      plot.sh		  run_plots.sh
e1	   HHSimulator_long_batch.py  plot_simulation.py
e2	   HHSimulator_long.py	      run_HH_long.sh


In [6]:
def load_db_numpy(A_list, fc_list, data_dir='../NIEMRE_Feb20_2025/Hodgkin_Huxley/data'):
    x_list, y_list, pt_list = [], [], []
    t_indices_list, t_values_list, t_params_list = [], [], []

    total_steps = len(A_list) * len(fc_list)
    step = 0

    for A_ID in A_list:
        for fc_ID in fc_list:
            step += 1
            if step % 25 == 0 or step == 1 or step == total_steps:
                print(f"[{step}/{total_steps}] Processing A_ID={A_ID}, fc_ID={fc_ID}")

            base = f"A_{A_ID}_fc{fc_ID}"

            files = {
                'x': f'x_{base}.pkl',
                'y': f'y_{base}.pkl',
                'pt': f'pt_{base}.pkl',
                't_indices': f't_indices_{base}.pkl',
                't_values': f't_values_{base}.pkl',
                't_params': f't_params_{base}.pkl',
            }

            paths = {k: os.path.join(data_dir, v) for k, v in files.items()}
            data = {k: load_file(v) for k, v in paths.items()}

            if all(v is not None for v in data.values()):
                x_list.append(np.array(data['x']))
                y_list.append(np.array(data['y']))
                pt_list.append(np.array(data['pt']))
                t_indices_list.append(np.array(data['t_indices']))
                t_values_list.append(np.array(data['t_values']))
                t_params_list.append(np.array(data['t_params']))
            else:
                print(f"❌ Missing data for A_ID={A_ID}, fc_ID={fc_ID} — skipping")

    x = np.stack(x_list)
    y = np.stack(y_list)
    pt = np.stack(pt_list)
    t_indices = np.stack(t_indices_list)
    t_values = np.stack(t_values_list)
    t_params = np.stack(t_params_list)

    total_samples = x.shape[0]
    mask, nan_samples_count, large_samples_count = get_mask_numpy(x, y)

    print(f"\n✅ Total samples before filtering: {total_samples}")
    print(f"🚫 Samples containing NaN: {nan_samples_count}")
    print(f"⚠️ Samples exceeding threshold {threshold}: {large_samples_count}")

    # Apply mask
    x = x[mask]
    y = y[mask]
    pt = pt[mask]
    t_indices = t_indices[mask]
    t_values = t_values[mask]
    t_params = t_params[mask]

    print(f"✅ Total samples after filtering: {x.shape[0]}")
    return x, y, pt, t_indices, t_values, t_params

In [7]:
# A_list = list(range(1, 100))
# fc_list = list(range(1, 200))
A_list = list(range(1, 20))
fc_list = list(range(1, 20))
# For NumPy version
x, y, pt, ti, tv, tp = load_db_numpy(A_list, fc_list)

[1/361] Processing A_ID=1, fc_ID=1
[25/361] Processing A_ID=2, fc_ID=6
[50/361] Processing A_ID=3, fc_ID=12
[75/361] Processing A_ID=4, fc_ID=18
[100/361] Processing A_ID=6, fc_ID=5
[125/361] Processing A_ID=7, fc_ID=11
[150/361] Processing A_ID=8, fc_ID=17
[175/361] Processing A_ID=10, fc_ID=4
[200/361] Processing A_ID=11, fc_ID=10
[225/361] Processing A_ID=12, fc_ID=16
[250/361] Processing A_ID=14, fc_ID=3
[275/361] Processing A_ID=15, fc_ID=9
[300/361] Processing A_ID=16, fc_ID=15
[325/361] Processing A_ID=18, fc_ID=2
[350/361] Processing A_ID=19, fc_ID=8
[361/361] Processing A_ID=19, fc_ID=19


ValueError: all input arrays must have the same shape

In [ ]:
x

In [ ]:
u_train,u_test, y_train, y_test, p_train, p_test = train_test_split(
    u, y, p, test_size=0.3, random_state=random_state)
train_intervals = [i * u.shape[2] for i in range(p_train.shape[0] + 1)]
test_intervals = [i * u.shape[2] for i in range(p_test.shape[0] + 1)]
data_list = [u_train,u_test, y_train, y_test]

In [ ]:
train_intervals[-1]

In [ ]:
test_intervals[-1]

In [ ]:
# for i, (u_, y_, p_) in enumerate(zip(u_train, y_train, p_train)):
#     if i >= 20:  # Show only first 5 plots
#         break
#     plt.figure(figsize=(10,1.5))
#     plt.plot(u_[3,:1000],label='Injected Current')
#     plt.plot(y_[0, :1000],label='True Voltage')
#     plt.legend(fontsize='x-small')
#     plt.title(f'Sample A={p_[0]},fc={p_[1]},fm={p_[2]}')
#     #plt.ylim([-0.1,1.1])
#     plt.show()

In [ ]:
for i, d in enumerate(data_list):
    # Reshape from (samples, features, time) -> (time, samples, features)
    d = d.transpose(0, 2, 1)
    
    # Flatten (time, samples, features) -> (time * samples, features)
    d = d.reshape(d.shape[0] * d.shape[1], d.shape[2])
    
    
    # Assign back to the list
    data_list[i] = d  

    # Optional: Print the shape for debugging
    print(f"Updated shape of data_list[{i}]: {d.shape}")

In [ ]:
u_train,u_test, y_train, y_test = data_list[0], data_list[1], data_list[2], data_list[3]  

In [ ]:
# for i in range(len(train_intervals) - 1):
#     start, end = train_intervals[i], train_intervals[i + 1]
#     plt.figure(figsize=(10,1.5))
#     plt.plot(u_train[start:end,3],label='Injected Current')
#     plt.plot(y_train[start:end],label='True Voltage')
#     plt.legend(fontsize='x-small')
#     plt.title(f'Sample A={p_train[i][0]},fc={p_train[i][1]},fm={p_train[i][2]}')
#     #plt.ylim([-0.1,1.1])
#     plt.show()

#### 2) Instantiate, train & test the network
Parameters are mostly the same as in Herbert Jaeger's original Matlab code. 

In [ ]:
esn = ESN(n_inputs = u.shape[1],
          n_outputs = y.shape[1],
          n_reservoir = 1000,
          spectral_radius = 0.25,
          sparsity = 0.95,
          noise = 0.001,
#           input_shift = [0,0],
#           input_scaling = [0.01, 3],
#           teacher_scaling = 1.12,
#           teacher_shift = -0.7,
#           out_activation = np.tanh,
#           inverse_out_activation = np.arctanh,
          random_state = random_state,
          silent = False)

pred_train = esn.fit(u_train,y_train)

print("test error:")
pred_test = esn.predict(u_test)
print(np.sqrt(np.mean((pred_test - y_test)**2)))

#### 3) Plots
First, a look at the control signal, the target signal and the output of the model both during training and during testing.

In [ ]:
# Ensure the 'plots' directory exists
#plots_dir = "plots"
plots_dir = os.path.join("plots", "reservoir N = 2000, noise = .01")
os.makedirs(plots_dir, exist_ok=True)
SAVEFIGS = True

In [ ]:
for i in range(len(train_intervals) - 1):
    start, end = train_intervals[i], train_intervals[i + 1]
    plt.figure(figsize=(10,1.5))
    plt.plot(u_train[start:end,3],label='Injected Current')
    plt.plot(y_train[start:end],label='True Voltage')
    plt.legend(fontsize='x-small')
    plt.title(f'Train A={p_train[i][0]},fc={p_train[i][1]},fm={p_train[i][2]}')
    if SAVEFIGS:
        filename = os.path.join(plots_dir, f'train_data_{i}_A{p_train[i][0]}_fc{p_train[i][1]}_fm{p_train[i][2]}.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for i in range(len(test_intervals) - 1):
    start, end = test_intervals[i], test_intervals[i + 1]
    plt.figure(figsize=(10,1.5))
    plt.plot(u_test[start:end,3],label='Injected Current')
    plt.plot(y_test[start:end],label='True Voltage')
    plt.legend(fontsize='x-small')
    plt.title(f'Test A={p_test[i][0]},fc={p_test[i][1]},fm={p_test[i][2]}')
    # Save the figure
    if SAVEFIGS:
        filename = os.path.join(plots_dir, f'test_data_{i}_A{p_train[i][0]}_fc{p_train[i][1]}_fm{p_train[i][2]}.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for i in range(len(train_intervals) - 1):
    start, end = train_intervals[i], train_intervals[i + 1]
    plt.figure(figsize=(10,1.5))
    plt.plot(u_train[start:end,3],label='Injected Current')
    plt.plot(y_train[start:end],label='True Voltage')
    plt.plot(pred_train[start:end],label='Predicted Voltage')
    plt.legend(fontsize='x-small')
    plt.title(f'training A={p_train[i][0]},fc={p_train[i][1]},fm={p_train[i][2]}')
    # Save the figure
    if SAVEFIGS:
        filename = os.path.join(plots_dir, f'train_model_{i}_A{p_train[i][0]}_fc{p_train[i][1]}_fm{p_train[i][2]}.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for i in range(len(test_intervals) - 1):
    start, end = test_intervals[i], test_intervals[i + 1]
    plt.figure(figsize=(10,1.5))
    plt.plot(u_test[start:end,3],label='Injected Current')
    plt.plot(y_test[start:end],label='True Voltage')
    plt.plot(pred_test[start:end],label='Predicted Voltage')
    plt.legend(fontsize='x-small')
    plt.title(f'testing A={p_test[i][0]},fc={p_test[i][1]},fm={p_test[i][2]}')
    if SAVEFIGS:
        filename = os.path.join(plots_dir, f'test_model_{i}_A{p_train[i][0]}_fc{p_train[i][1]}_fm{p_train[i][2]}.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

What we see is that we can't see much, except that the amplitude is systematically too small (any ideas why that is?).

So let's look at a few spectrograms to see how the frequency spectrum of these signals changes over time.

In [ ]:
def draw_spectogram(data):
    plt.specgram(data,Fs=4,NFFT=256,noverlap=150,cmap=plt.cm.bone,detrend=lambda x:(x-0.5))
    plt.gca().autoscale('x')
    plt.ylim([0,0.5])
    plt.ylabel("freq")
    plt.yticks([])
    plt.xlabel("time")
    plt.xticks([])

plt.figure(figsize=(7,1.5))
draw_spectogram(y_train.flatten())
plt.title("training: target")
if SAVEFIGS:
    filename = os.path.join(plots_dir, f'train_target_spectogram.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()
plt.figure(figsize=(7,1.5))
draw_spectogram(pred_train.flatten())
plt.title("training: model")
if SAVEFIGS:
    filename = os.path.join(plots_dir, f'train_model_spectogram.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(3,1.5))
draw_spectogram(y_test.flatten())
plt.title("test: target")
if SAVEFIGS:
    filename = os.path.join(plots_dir, f'test_target_spectogram.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()
plt.figure(figsize=(3,1.5))
draw_spectogram(pred_test.flatten())
plt.title("test: model")
if SAVEFIGS:
    filename = os.path.join(plots_dir, f'test_model_spectogram.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()

It's a frequency generator!